In [280]:
import os
import pandas as pd
from pprint import pprint
import json

In [281]:
os.getcwd()

'c:\\Users\\Lukeg\\Desktop\\Capstone Virtual Environment\\Capstone_Project_AIM\\LeafletJS\\JSON'

In [282]:
from pathlib import Path

# Simple: csv files folder is in the same directory as this notebook
csv_folder = Path("csv files")

# List all CSV files
csv_files = sorted([f.name for f in csv_folder.glob('*.csv')])

print(f"📄 Found {len(csv_files)} CSV files: {csv_files}")

📄 Found 5 CSV files: ['A1.csv', 'B1.csv', 'H1.csv', 'M1.csv', 'M2.csv']


In [283]:
print(csv_files)

['A1.csv', 'B1.csv', 'H1.csv', 'M1.csv', 'M2.csv']


In [284]:
building_names = {
    "A": "Building A",
    "B": "Building B",
    "C": "Building C",
    "D": "Building D",
    "E": "Building E",
    "F": "F Building",
    "G": "Building G",
    "H": "Building H",
    "J": "Building J",
    "K": "Building K",
    "M": "Building M",
    "T": "Building T",
}

In [285]:
all_docs = {}
for file in csv_files:
    # Read from the csv_folder path
    file_path = csv_folder / file
    doc = pd.read_csv(file_path)
    
    name = file.replace(".csv", "")
    
    building_code = name[0]
    floor_number = name[1:]
    
    building_name = building_names[building_code]
    if building_name not in all_docs:
        all_docs[building_name] = {}
    all_docs[building_name][floor_number] = doc

print(f"✅ Loaded {len(all_docs)} buildings")

✅ Loaded 4 buildings


In [286]:
object_types = ["room", "building_connection", "stairs", "elevator", "outside_exit", "bathroom", "eatery"]
connects_floorplans = ["stairs", "elevator", "building_connection"]

In [287]:
accepted_node_types = ["object", "turn", "intersection", "roundabout"]

In [288]:
def object_logic(row, rooms, exits):
    object_type = row['Object_category_type']
    if not isinstance(object_type, str):
        return rooms, exits
    object_type = object_type.lower().replace(" ", "")
    object_type_list = object_type.split(",")

    objects = row['Object']
    objects = objects.replace(" ", "")
    object_list = objects.split(",")

    doors = row['Door']
    if isinstance(doors, str):
        doors = doors.replace(" ", "")
        door_list = doors.split(",")
        
    
    exit_conns = row['Exit_Connections']
    exit_conns_list = []
    if isinstance(exit_conns, str):
        exit_conns = exit_conns.replace(" ", "")
        exit_conns_raw = exit_conns.split(",")
        for element in exit_conns_raw:
            clean = element.strip("[]")
            exit_entries = clean.split("|")
            exit_conns_list.append(exit_entries)
            
    
    door_counter = 0
    exits_counter = 0
    for i,entry in enumerate(object_type_list):
        selected_obj = object_list[i]
        if entry == "room":
            door = door_list[door_counter]

            if selected_obj not in rooms:
                rooms[selected_obj] = []
            rooms[selected_obj].append(door)
            door_counter += 1
        elif entry == "exit":
            # print("exit")
            # print(object_type_list)
            # print(exit_conns_list)
            exit_destination = exit_conns_list[exits_counter]
            if selected_obj not in exits:
                exits[selected_obj] = []
            exits[selected_obj].extend(exit_destination)
            exits_counter += 1

    return rooms, exits

In [289]:
def split_csv(value):
    if isinstance(value, str):
        return value.replace(" ", "").split(",")
    return []


def parse_exit_conns(value):
    if not isinstance(value, str):
        return []
    cleaned = value.replace(" ", "")
    return [item.strip("[]").split("|") for item in cleaned.split(",")]


def parse_vertical_conns(value):
    """Parse vertical connections in format: [node1|node2|node3]
    Returns a single list of connector IDs from the bracket notation.
    """
    # Handle pandas NaN values
    if pd.isna(value):
        return []
    if not isinstance(value, str) or not value:
        return []
    # Remove spaces and brackets, then split by |
    cleaned = value.replace(" ", "").strip("[]")
    if not cleaned:
        return []
    return cleaned.split("|")


def node_navigation(row, navigationGraph, roomToNode, exitToNode):

    node = row["Node"]
    connections = split_csv(row["Node_connections"])
    object_ids = split_csv(row["Object"])
    node_types = split_csv(str(row["Node_type"]).lower())
    object_types = split_csv(str(row["Object_type"]).lower())
    categories = split_csv(str(row["Object_category_type"]).lower())
    doors = split_csv(row["Door"])
    exit_conns = parse_exit_conns(row["Exit_Connections"])
    
    # 🆕 Handle vertical connections (stairs/elevators between floors)
    vertical_value = row.get("Vertical_node_connections", pd.NA)
    vertical_conns = parse_vertical_conns(vertical_value)

    if node not in navigationGraph:
        navigationGraph[node] = {"connections": connections, "represents": []}

    represents = navigationGraph[node]["represents"]

    # Map short names → proper names
    SIMPLE_TYPES = {
        "int": "intersection",
        "turn": "turn",
        "roundabout": "roundabout"
    }

    # ------------------------------------------------------------
    # 1. Add simple node types FIRST (turn, intersection, etc.)
    # ------------------------------------------------------------
    has_only_simple = True
    for nt in node_types:
        if nt in SIMPLE_TYPES:
            represents.append({"type": SIMPLE_TYPES[nt]})
        else:
            has_only_simple = False

    # If node is ONLY a simple type with no objects, stop here
    if has_only_simple and not object_types:
        return navigationGraph, roomToNode, exitToNode

    # ------------------------------------------------------------
    # 2. Now add objects (room, stairs, exit, bathroom, etc.)
    # ------------------------------------------------------------
    door_idx = 0
    exit_idx = 0
    vertical_idx = 0

    for obj_type, obj_id, cat in zip(object_types, object_ids, categories):

        rep = {"type": obj_type, "id": obj_id}

        if cat == "room":
            if door_idx < len(doors):
                rep["door"] = doors[door_idx]
                door_idx += 1
            
            # 🔧 UPDATED: Store rooms as arrays for multi-node support
            if obj_id not in roomToNode:
                roomToNode[obj_id] = []
            roomToNode[obj_id].append(node)

        elif cat == "exit":
            if exit_idx < len(exit_conns):
                rep["goesTo"] = exit_conns[exit_idx]
                exit_idx += 1
            else:
                rep["goesTo"] = None
            
            # 🆕 Add vertical connections for stairs/elevators
            # verticalConnections is a flat list from [connector1|connector2|...]
            if obj_type in ["stairs", "elevator"] and vertical_conns:
                rep["verticalConnections"] = vertical_conns
            
            # 🔧 UPDATED: Store exits as arrays for multi-node support
            if obj_id not in exitToNode:
                exitToNode[obj_id] = []
            exitToNode[obj_id].append(node)

        represents.append(rep)

    return navigationGraph, roomToNode, exitToNode


In [290]:
#print(doc.columns)
def floorplan_json_generator(doc):
    navigationGraph = {}
    roomToNode = {}
    entranceToNode = {}
    objects = {
        'rooms' : None,
        'exits' : None
              }
    rooms = {}
    exits = {}
    for index, row in doc.iterrows():

        rooms, exits = object_logic(row, rooms, exits)
        navigationGraph, roomToNode, exitToNode = node_navigation(row, navigationGraph, roomToNode, entranceToNode)


    # #Sorting to make consistent
    # navigationGraph = dict(sorted(
    #     navigationGraph.items(),
    #     key=lambda x: int(x[0].split("_")[1])
    #     if "_" in x[0] and x[0].split("_")[1].isdigit()
    #     else float('inf')
    # ))
    
    objects['rooms'] = rooms
    objects['exits'] = exits
    return navigationGraph, roomToNode, entranceToNode, objects
    

In [291]:
def extract_vertical_connectors(navigationGraph):
    """Extract vertical connectors (stairs/elevators) from navigation graph
    Returns a dictionary where keys are connector IDs (e.g., 'Stairs_1_M2')
    and values contain the connector info with verticalConnections as a dict
    mapping floor numbers to connector IDs on those floors.
    """
    vertical_connectors = {}
    
    for node_id, node_info in navigationGraph.items():
        if "represents" not in node_info:
            continue
            
        for rep in node_info["represents"]:
            # Look for stairs or elevators with verticalConnections
            if rep.get("type") in ["stairs", "elevator"] and "verticalConnections" in rep:
                connector_id = rep["id"]
                
                # Build verticalConnections as a dictionary: {floor_number: connector_id}
                vertical_connections_dict = {}
                goes_to = rep.get("goesTo", [])
                vert_conns = rep.get("verticalConnections", [])
                
                # Match each floor in goesTo with corresponding connector in verticalConnections
                for i, floor_dest in enumerate(goes_to):
                    if i < len(vert_conns):
                        # Extract floor number from destination (e.g., "M2" -> "2")
                        # Assumes format like "M2", "H1", etc.
                        floor_num = floor_dest[-1] if floor_dest else None
                        if floor_num:
                            vertical_connections_dict[floor_num] = vert_conns[i]
                
                vertical_connectors[connector_id] = {
                    "id": connector_id,
                    "type": rep["type"],
                    "nodeId": node_id,
                    "goesTo": goes_to,
                    "verticalConnections": vertical_connections_dict
                }
    
    return vertical_connectors

In [292]:
import json

node_data = {}
for building, data in all_docs.items():
    building_code = building[-1]
    building_path = f"Floorplans/{building}"
    for floor in data:
        floorplan_code = f"{building_code}{floor}"
        svg_file = f"{floorplan_code}.svg"
        doc = data[floor]
        nav_graph, room_to_node, exit_to_node, objects = floorplan_json_generator(doc)
        
        # Extract vertical connectors (stairs/elevators)
        vertical_connectors = extract_vertical_connectors(nav_graph)
        
        if building not in node_data:
            node_data[building] = {"path": building_path,
                                   "floors" : {}
                                  }
        
        floor_data = {
            "plan": svg_file,
            "navigationGraph": nav_graph,
            "roomToNode": room_to_node,
            "exitToNode": exit_to_node,
            "objects": objects
        }
        
        # Only add verticalConnectors if there are any
        if vertical_connectors:
            floor_data["verticalConnectors"] = vertical_connectors
        
        node_data[building]["floors"][floor] = floor_data

# Output to current directory (JSON folder)
with open("all_node_data.json", "w") as f:
    json.dump(node_data, f, indent=4)

print(f"✅ Successfully wrote all_node_data.json")

✅ Successfully wrote all_node_data.json


In [293]:
pprint(node_data)

{'Building A': {'floors': {'1': {'exitToNode': {},
                                 'navigationGraph': {},
                                 'objects': {'exits': {}, 'rooms': {}},
                                 'plan': 'A1.svg',
                                 'roomToNode': {}}},
                'path': 'Floorplans/Building A'},
 'Building B': {'floors': {'1': {'exitToNode': {},
                                 'navigationGraph': {},
                                 'objects': {'exits': {}, 'rooms': {}},
                                 'plan': 'B1.svg',
                                 'roomToNode': {}}},
                'path': 'Floorplans/Building B'},
 'Building H': {'floors': {'1': {'exitToNode': {'Elevator_H1': ['H1_3'],
                                                'F-Building_H1': ['H1_17'],
                                                'M-Building_H1': ['M1_entry'],
                                                'Outside-Exit_1_H1': ['H1_4'],
                           

In [294]:
# Verification: Check if vertical connections were added
print("\n🔍 Verifying Vertical Connections:")
print("=" * 60)

for building, building_data in node_data.items():
    for floor, floor_data in building_data["floors"].items():
        nav_graph = floor_data["navigationGraph"]
        
        for node_id, node_info in nav_graph.items():
            for rep in node_info.get("represents", []):
                if rep.get("type") in ["stairs", "elevator"]:
                    vert_conns = rep.get("verticalConnections")
                    if vert_conns:
                        print(f"✅ {building} Floor {floor}: {rep['id']}")
                        print(f"   → verticalConnections: {vert_conns}")
                    else:
                        print(f"⚠️  {building} Floor {floor}: {rep['id']} (NO vertical connections)")

print("=" * 60)


🔍 Verifying Vertical Connections:
✅ Building H Floor 1: Elevator_H1
   → verticalConnections: ['Elevator_H2', 'Elevator_H3']
✅ Building H Floor 1: Stairs_1_H1
   → verticalConnections: ['Stairs_1_H2', 'Stairs_1_H3']
✅ Building H Floor 1: Stairs_2_H1
   → verticalConnections: ['Stairs_2_H2', 'Stairs_2_H3']
✅ Building H Floor 1: Stairs_3_H1
   → verticalConnections: ['Stairs_3_H2', 'Stairs_3_H3']
✅ Building M Floor 1: Stairs_1_M1
   → verticalConnections: ['Stairs_1_M2', 'Stairs_1_M3']
✅ Building M Floor 1: Elevator_M1
   → verticalConnections: ['Elevator_M2', 'Elevator_M3']
✅ Building M Floor 1: Stairs_2_M1
   → verticalConnections: ['Stairs_2_M2', 'Stairs_2_M3']
✅ Building M Floor 1: Stairs_3_M1
   → verticalConnections: ['Stairs_3_M2', 'Stairs_3_M3']
✅ Building M Floor 2: Stairs_1_M2
   → verticalConnections: ['Stairs_1_M1', 'Stairs_1_M3']
✅ Building M Floor 2: Elevator_M2
   → verticalConnections: ['Elevator_M1', 'Elevator_M3']
✅ Building M Floor 2: Stairs_2_M2
   → verticalConnecti

In [295]:
# Check if verticalConnectors is at the floor level
print("🔍 Checking verticalConnectors structure:")
print("=" * 60)

# Check Building M Floor 1
if "Building M" in node_data:
    m_floor_1 = node_data["Building M"]["floors"]["1"]
    print(f"Keys in Building M Floor 1: {list(m_floor_1.keys())}")
    
    if "verticalConnectors" in m_floor_1:
        print(f"\n✅ verticalConnectors found at floor level!")
        print(f"Number of vertical connectors: {len(m_floor_1['verticalConnectors'])}")
        print(f"\nVertical connectors on M Floor 1:")
        for conn in m_floor_1['verticalConnectors']:
            print(f"  - {conn['id']} ({conn['type']}) at node {conn['nodeId']}")
            print(f"    Goes to: {conn['goesTo']}")
            print(f"    Connects to: {conn['verticalConnections']}")
    else:
        print("❌ verticalConnectors NOT found at floor level")
        
print("=" * 60)

🔍 Checking verticalConnectors structure:
Keys in Building M Floor 1: ['plan', 'navigationGraph', 'roomToNode', 'exitToNode', 'objects', 'verticalConnectors']

✅ verticalConnectors found at floor level!
Number of vertical connectors: 4

Vertical connectors on M Floor 1:


TypeError: string indices must be integers

In [ ]:
# Check where the file is being written
import os
print(f"Current working directory: {os.getcwd()}")
print(f"File will be written to: {os.path.abspath('all_node_data.json')}")

# Check if file exists and when it was last modified
if os.path.exists('all_node_data.json'):
    import datetime
    mod_time = os.path.getmtime('all_node_data.json')
    mod_datetime = datetime.datetime.fromtimestamp(mod_time)
    print(f"File exists! Last modified: {mod_datetime}")
    
    # Read first few lines to check content
    with open('all_node_data.json', 'r') as f:
        content = f.read(500)
        if 'verticalConnectors' in content:
            print("✅ File contains 'verticalConnectors'")
        else:
            print("❌ File does NOT contain 'verticalConnectors'")
else:
    print("❌ File does not exist at this path!")

Current working directory: c:\Users\Lukeg\Desktop\Capstone Virtual Environment\Capstone_Project_AIM\LeafletJS\JSON
File will be written to: c:\Users\Lukeg\Desktop\Capstone Virtual Environment\Capstone_Project_AIM\LeafletJS\JSON\all_node_data.json
File exists! Last modified: 2025-12-08 20:34:23.092826
❌ File does NOT contain 'verticalConnectors'


In [ ]:
# Check Floor 2 for Stairs_3_M2
print("🔍 Checking Building M Floor 2:")
print("=" * 60)

if "Building M" in node_data and "2" in node_data["Building M"]["floors"]:
    m_floor_2 = node_data["Building M"]["floors"]["2"]
    
    if "verticalConnectors" in m_floor_2:
        print(f"✅ verticalConnectors keys on Floor 2: {list(m_floor_2['verticalConnectors'].keys())}")
        
        if "Stairs_3_M2" in m_floor_2['verticalConnectors']:
            stairs_3 = m_floor_2['verticalConnectors']['Stairs_3_M2']
            print(f"\n✅ Found Stairs_3_M2!")
            print(f"   ID: {stairs_3['id']}")
            print(f"   Type: {stairs_3['type']}")
            print(f"   Node: {stairs_3['nodeId']}")
            print(f"   Goes to: {stairs_3['goesTo']}")
            print(f"   Vertical connections: {stairs_3['verticalConnections']}")
        else:
            print("❌ Stairs_3_M2 NOT found in verticalConnectors!")
    
    if "exitToNode" in m_floor_2:
        print(f"\n📋 exitToNode keys on Floor 2: {list(m_floor_2['exitToNode'].keys())}")
        if "Stairs_3_M2" in m_floor_2['exitToNode']:
            print(f"   ✅ Stairs_3_M2 → {m_floor_2['exitToNode']['Stairs_3_M2']}")
        else:
            print("   ❌ Stairs_3_M2 NOT in exitToNode!")

print("=" * 60)